In [18]:
import numpy as np
import periodictable
import sys
sys.path.append('..')
import units
import read_parameters

In [2]:
units.Dalton

1.1531805312624011e-21

In [29]:
# structure.py

def read_xyz_file(xyz_file):
    xyz = open(xyz_file).readlines()
    
    structure = {}
    structure['n_atoms'] = int(xyz[0].split()[0])
    try:
        structure['current_step'] = int(xyz[1].split()[0])
        structure['total_energy'] = float(xyz[1].split()[1])
    except:
        pass
    
    element_list = []
    mass_list = []
    str_x_list = []
    str_y_list = []
    str_z_list = []
    str_vx_list = []
    str_vy_list = []
    str_vz_list = []
    str_fx_list = []
    str_fy_list = []
    str_fz_list = []
    for i_atom in range(structure['n_atoms']):
        element_list.append(str(xyz[i_atom+2].split()[0]))
        str_x_list.append(float(xyz[i_atom+2].split()[1]))
        str_y_list.append(float(xyz[i_atom+2].split()[2]))
        str_z_list.append(float(xyz[i_atom+2].split()[3]))
        try:
            str_vx_list.append(float(xyz[i_atom+2].split()[4]))
            str_vy_list.append(float(xyz[i_atom+2].split()[5]))
            str_vz_list.append(float(xyz[i_atom+2].split()[6]))
        except:
            str_vx_list.append(0.0)
            str_vy_list.append(0.0)
            str_vz_list.append(0.0)
            pass
            
        try:
            str_fx_list.append(float(xyz[i_atom+2].split()[7]))
            str_fy_list.append(float(xyz[i_atom+2].split()[8]))
            str_fz_list.append(float(xyz[i_atom+2].split()[9]))
        except:
            pass
            
    structure['elements'] = element_list
    for i_ele in element_list:
        mass_list.append(periodictable.elements.symbol(i_ele).mass)
    structure['masses'] = np.array(mass_list) * units.Dalton
    
    str_x_list = np.array(str_x_list)
    str_y_list = np.array(str_y_list)
    str_z_list = np.array(str_z_list)
    structure['coordinates'] = np.array([str_x_list, str_y_list, str_z_list]).T
    
    str_vx_list = np.array(str_vx_list)
    str_vy_list = np.array(str_vy_list)
    str_vz_list = np.array(str_vz_list)
    structure['velocities'] = np.array([str_vx_list, str_vy_list, str_vz_list]).T
    
    acceleration_list = []
    if len(str_fx_list) > 0:
        str_fx_list = np.array(str_fx_list)
        str_fy_list = np.array(str_fy_list)
        str_fz_list = np.array(str_fz_list)
        structure['forces'] = np.array([str_fx_list, str_fy_list, str_fz_list]).T
        structure['accelerations'] = np.array([structure['forces'][i_atom]/structure['masses'][i_atom] \
                                          for i_atom in range(len(structure['forces']))])
    
    return structure

def write_xyz_file(xyz_file, structure_dict):
    xyz_file.write(str(structure_dict['n_atoms'])+'\n')
    xyz_file.write(str(structure_dict['current_step'])+'    '+str(structure_dict['total_energy'])+'\n')
    for i_atom in range(structure_dict['n_atoms']):
        xyz_file.write(structure_dict['elements'][i_atom]+'    ')
        for i in range(3):
            xyz_file.write(str(structure_dict['coordinates'][i_atom][i])+'    ')
        for i in range(3):
            xyz_file.write(str(structure_dict['velocities'][i_atom][i])+'    ')
        for i in range(3):
            xyz_file.write(str(structure_dict['forces'][i_atom][i])+'    ')
        xyz_file.write('\n')

In [20]:
print(read_xyz_file('MoREST.str'))

{'n_atoms': 2, 'elements': ['Al', 'F'], 'masses': array([3.11145843e-20, 2.19085887e-20]), 'coordinates': array([[0.1, 0.2, 0.3],
       [1.1, 1.2, 1.3]]), 'velocities': array([[0., 0., 0.],
       [0., 0., 0.]])}


In [7]:
# potential.py

class ml_potential:
    def __init__(self):
        return None
        
    def read_potential_force(self, structure):
        return np.random.random_sample(), np.random.rand(2,3)

In [58]:
# phase_space_sampling.py

class velocity_Verlet:
    '''
    This class implements velocity Verlet algorithm to do microcanonical ensemble (NVE MD) sampling.
    MoREST.traj records the trajectory in an extended xyz format:
    ------------------------------------
    n_atoms
    current_step    total_energy
    element1    x1    y1    z1    velocity_x1    velocity_y1    velocity_z1    force_x1    force_y1    force_z1
    ...
    ------------------------------------
    MoREST.str records the initial xyz structure of the system
    MoREST.str_new records the current xyz structure of the system
    '''
    
    def __init__(self, sampling_parameters, md_parameters):
        #self.md_parameters = np.load('MoREST_md_parameters.npy',allow_pickle=True).item()
        self.sampling_parameters = sampling_parameters
        self.md_parameters = md_parameters
        
        self.structure = self.get_current_structure()
        
        if self.sampling_parameters['sampling_restart']:
            self.__log_traj = open('MoREST.traj','a')
        else:
            self.__log_traj = open('MoREST.traj','w')
            write_xyz_file(self.__log_traj, self.structure)
        
        
    def generate_new_step(self):
        time_step = self.md_parameters['md_time_step']
        self.next_structure = {}
        
        self.next_structure['n_atoms'] = self.structure['n_atoms']
        self.next_structure['elements'] = self.structure['elements']
        self.next_structure['current_step'] = self.structure['current_step'] + 1
        self.next_structure['masses'] = self.structure['masses']
        self.next_structure['coordinates'] = self.structure['coordinates'] \
                                           + self.structure['velocities'] * time_step \
                                           + 0.5 * self.structure['accelerations'] * time_step**2
        self.next_structure['total_energy'], self.next_structure['forces'] = \
                                           ml_potential().read_potential_force(self.next_structure['coordinates'])
        self.next_structure['accelerations'] = np.array([self.next_structure['forces'][i_atom]/self.next_structure['masses'][i_atom] \
                                              for i_atom in range(len(self.next_structure['forces']))])
        self.next_structure['velocities'] = self.structure['velocities'] \
                                          + 0.5 * (self.structure['accelerations'] + self.next_structure['accelerations']) * time_step
        
        self.structure = self.next_structure
        str_new = open('MoREST.str_new','w')
        write_xyz_file(str_new, self.structure)
        write_xyz_file(self.__log_traj, self.structure)
        
        return None
    
        
    def get_current_structure(self):
        if self.sampling_parameters['sampling_restart']:
            structure = read_xyz_file('MoREST.str_new')
        else:
            structure = read_xyz_file('MoREST.str')
            structure['current_step'] = 0
            structure['total_energy'], structure['forces'] = ml_potential().read_potential_force(structure['coordinates'])
            structure['accelerations'] = np.array([structure['forces'][i_atom] / structure['masses'][i_atom] \
                                              for i_atom in range(len(structure['forces']))])            
        return structure

In [51]:
periodictable.elements.symbol('Al').mass

26.981538

In [52]:
def phase_space_sampling(sampling_parameters, md_parameters):
    if sampling_parameters['sampling_method'].upper() in ['MD'] and sampling_parameters['sampling_ensemble'].upper() in ['NVE']:
        sampling_job = velocity_Verlet(sampling_parameters, md_parameters)
        current_structure = sampling_job.get_current_structure()
        max_time_step = int(md_parameters['md_simulation_time']/md_parameters['md_time_step']) + 1
        for i_step in range(current_structure['current_step']+1, max_time_step):
            sampling_job.generate_new_step()

# test

In [62]:
parameters = read_parameters.read_parameters(log_morest='MoREST.log', parameter_file='../MoREST.in')
sampling_parameters = parameters.get_sampling_parameters()
md_parameters = parameters.get_md_parameters()
phase_space_sampling(sampling_parameters, md_parameters)